<a href="https://colab.research.google.com/github/tryan01/gravitational-waves/blob/main/microquant_v1_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
import os

print("Authenticating with Google Cloud...")
auth.authenticate_user()

# Define the bucket and the specific archives to pull
bucket_name = "tryanheider-kalshi-bucket"
archives_to_download = [
    "raw_vault_2026-04-16.tar.gz",
    "raw_vault_2026-04-17.tar.gz",
    "raw_vault_2026-04-18.tar.gz",
    "raw_vault_2026-04-19.tar.gz",
    "raw_vault_2026-04-20.tar.gz",
]

# Create a local directory in Colab to hold the archives
download_dir = '/content/kalshi_raw_archives'
os.makedirs(download_dir, exist_ok=True)

print(f"Downloading archives from gs://{bucket_name} to {download_dir}...")

for archive in archives_to_download:
    gcs_path = f"gs://{bucket_name}/{archive}"
    local_path = os.path.join(download_dir, archive)

    # Use gsutil for fast, multi-threaded downloading
    print(f"Fetching {archive}...")
    !gcloud storage cp {gcs_path} {local_path}

print("\nDownload complete. Archives available on local Colab disk:")
print(os.listdir(download_dir))

In [ ]:
import os
import glob
import tarfile
import pandas as pd
import numpy as np
import concurrent.futures
from typing import Dict, List

# --- CONFIGURATION ---
DOWNLOAD_DIR = '/content/kalshi_raw_archives'
TARGET_ARCHIVES: List[str] = glob.glob(f"{DOWNLOAD_DIR}/*.tar.gz")
WORKING_DIR: str = '/content/kalshi_working_data'

# NEW: Dual-Output Architecture
OUTPUT_DENSE_DIR: str = '/content/kalshi_dense_state_v2'  # For the ML Model
OUTPUT_RAW_DIR: str = '/content/kalshi_clean_raw_v2'      # For the Zero-Assumption Simulator

LEVELS: int = 5
STRIDE_SEC: float = 1.0
WINDOW_HOURS: float = 4.0
WINDOW_SEC: float = WINDOW_HOURS * 3600.0

os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(OUTPUT_DENSE_DIR, exist_ok=True)
os.makedirs(OUTPUT_RAW_DIR, exist_ok=True)

def _extract_archive(archive_path: str, extract_path: str) -> None:
    """Extracts a tar archive safely."""
    if not os.path.exists(archive_path):
        print(f"WARNING: Archive not found at {archive_path}")
        return

    archive_name: str = os.path.basename(archive_path).replace('.tar', '').replace('.tar.gz', '')
    target_folder: str = os.path.join(extract_path, archive_name)

    if os.path.exists(target_folder):
        print(f"Archive {archive_name} already extracted. Skipping decompression.")
        return

    print(f"Extracting {archive_path} to {target_folder}...")
    os.makedirs(target_folder, exist_ok=True)
    with tarfile.open(archive_path, 'r') as tar:
        tar.extractall(path=target_folder)
    print(f"Finished extracting {archive_name}.")

def unify_timestamp(ts_series: pd.Series) -> pd.Series:
    """Forces any timestamp to a unified UNIX epoch float in seconds."""
    if pd.api.types.is_numeric_dtype(ts_series):
        return np.where(ts_series > 1e12, ts_series / 1000.0, ts_series).astype(float)
    dt = pd.to_datetime(ts_series, format='mixed', errors='coerce', utc=True)
    return dt.astype('int64') / 1e9

def _snapshot_state(bids: Dict[float, float], asks: Dict[float, float],
                    vol_yes: float, vol_no: float,
                    min_trade: float, max_trade: float,
                    current_ts: float, max_ts: float) -> Dict[str, float]:
    """Extracts top N levels and normalizes the state."""
    sorted_bids: List[tuple] = sorted(bids.items(), key=lambda x: x[0], reverse=True)
    sorted_asks: List[tuple] = sorted(asks.items(), key=lambda x: x[0])

    state: Dict[str, float] = {'ts': current_ts}

    for i in range(LEVELS):
        state[f'bid_px_{i+1}'] = float(sorted_bids[i][0]) if i < len(sorted_bids) else 0.0
        state[f'bid_vol_{i+1}'] = float(sorted_bids[i][1]) if i < len(sorted_bids) else 0.0
        state[f'ask_px_{i+1}'] = float(sorted_asks[i][0]) if i < len(sorted_asks) else 1.0
        state[f'ask_vol_{i+1}'] = float(sorted_asks[i][1]) if i < len(sorted_asks) else 0.0

    state['taker_vol_yes'] = vol_yes
    state['taker_vol_no'] = vol_no

    # NEW: Safely append the trade extremes (Will be np.inf/-np.inf if no trades occurred)
    state['window_min_trade_price'] = min_trade
    state['window_max_trade_price'] = max_trade

    state['time_fraction'] = (max_ts - current_ts) / WINDOW_SEC
    return state

def process_single_market(market_dir: str) -> str:
    """Processes a single market, cleans timestamps, saves Raw and Dense states."""
    ticker: str = os.path.basename(market_dir)
    dense_output_path: str = os.path.join(OUTPUT_DENSE_DIR, f"dense_{ticker}.parquet")
    raw_output_path = os.path.join(OUTPUT_RAW_DIR, f"clean_raw_{ticker}.parquet")

    if os.path.exists(dense_output_path) and os.path.exists(raw_output_path):
        return f"[{ticker}] SKIPPED: Both V2 files already exist."

    partition_files: List[str] = glob.glob(f"{market_dir}/*.parquet")
    if not partition_files:
        return f"[{ticker}] SKIPPED: No parquet files."

    try:
        # 1. Recreate perfect chronological reality
        df_list: List[pd.DataFrame] = [pd.read_parquet(f) for f in partition_files]
        master_df: pd.DataFrame = pd.concat(df_list, ignore_index=True)

        # 2. THE TIMESTAMP FIX
        if 'ts' in master_df.columns:
            master_df['ts'] = unify_timestamp(master_df['ts'])
        if 'local_recv_ts' in master_df.columns:
            master_df['local_recv_ts'] = unify_timestamp(master_df['local_recv_ts'])

        # 3. Sort chronologically by local arrival time
        master_df = master_df.sort_values(by=['local_recv_ts', 'seq']).reset_index(drop=True)

        # 4. Save Cleaned Raw Data for Simulator
        master_df.to_parquet(raw_output_path)

        # 5. Typecasting for L2 Walk Forward
        master_df['price_dollars'] = master_df['price_dollars'].astype(float)
        master_df['delta_fp'] = master_df['delta_fp'].astype(float)

        # 6. Time Grid Setup
        max_ts: float = master_df['local_recv_ts'].max()
        min_ts_clipped: float = max(master_df['local_recv_ts'].min(), max_ts - WINDOW_SEC)
        time_grid: np.ndarray = np.arange(min_ts_clipped, max_ts, STRIDE_SEC)

        resting_bids: Dict[float, float] = {}
        resting_asks: Dict[float, float] = {}
        grid_idx: int = 0
        grid_features: List[Dict[str, float]] = []
        rolling_taker_yes: float = 0.0
        rolling_taker_no: float = 0.0

        # NEW: Variables to track the extremes within the 1-second grid
        window_min_trade_price: float = np.inf
        window_max_trade_price: float = -np.inf

        last_snapshot_seq: int = -1

        # 7. Walk Forward from absolute t0
        for _, row in master_df.iterrows():
            current_event_ts: float = row['local_recv_ts']

            while grid_idx < len(time_grid) and current_event_ts > time_grid[grid_idx]:
                grid_features.append(_snapshot_state(
                    resting_bids, resting_asks,
                    rolling_taker_yes, rolling_taker_no,
                    window_min_trade_price, window_max_trade_price,
                    time_grid[grid_idx], max_ts
                ))
                grid_idx += 1

                # RESET window aggregates the millisecond we cross into the next second
                rolling_taker_yes, rolling_taker_no = 0.0, 0.0
                window_min_trade_price, window_max_trade_price = np.inf, -np.inf

            # --- THE FIX: Split extraction by Event Type ---

            if row['event_type'] in ['orderbook_snapshot', 'orderbook_delta']:
                is_yes = row['side'] == 'yes'
                price = float(row['price_dollars']) if is_yes else round(1.0 - float(row['price_dollars']), 2)

                book: Dict[float, float] = resting_bids if is_yes else resting_asks

                if row['event_type'] == 'orderbook_snapshot':
                    if row['seq'] != last_snapshot_seq:
                        resting_bids.clear()
                        resting_asks.clear()
                        last_snapshot_seq = row['seq']

                new_vol: float = book.get(price, 0.0) + (row['delta_fp'] if pd.notna(row['delta_fp']) else 0.0)
                if new_vol > 0.5:
                    book[price] = new_vol
                else:
                    book.pop(price, None)

            elif row['event_type'] == 'trade':
                # Use trade-specific columns
                is_yes = row['taker_side'] == 'yes'

                # Safely grab the traded price (Kalshi usually provides yes_price_dollars for all trades)
                if pd.notna(row.get('yes_price_dollars')):
                    price = float(row['yes_price_dollars'])
                elif pd.notna(row.get('no_price_dollars')):
                    price = round(1.0 - float(row['no_price_dollars']), 2)
                else:
                    continue # Skip if completely malformed

                # Safely parse the count
                trade_count = float(row['count_fp']) if pd.notna(row.get('count_fp')) else 0.0

                if is_yes:
                    rolling_taker_yes += trade_count
                else:
                    rolling_taker_no += trade_count

                # Update the extremes for the Triple Barrier method
                if price < window_min_trade_price:
                    window_min_trade_price = price
                if price > window_max_trade_price:
                    window_max_trade_price = price

        # Catch trailing time grid
        while grid_idx < len(time_grid):
            grid_features.append(_snapshot_state(
                resting_bids, resting_asks,
                rolling_taker_yes, rolling_taker_no,
                window_min_trade_price, window_max_trade_price, # NEW: Inject bounds
                time_grid[grid_idx], max_ts
            ))
            grid_idx += 1
            # Reset not strictly needed here as we are trailing, but good practice
            rolling_taker_yes, rolling_taker_no = 0.0, 0.0
            window_min_trade_price, window_max_trade_price = np.inf, -np.inf

        # 8. Save Dense Data for Model
        df_grid: pd.DataFrame = pd.DataFrame(grid_features)
        if not df_grid.empty:
            df_grid.to_parquet(dense_output_path, index=False)
            return f"[{ticker}] SUCCESS: Wrote Clean Raw & {len(df_grid)} Dense rows."
        else:
            return f"[{ticker}] FAILED: Output grid empty."

    except Exception as e:
        return f"[{ticker}] ERROR: {str(e)}"

if __name__ == '__main__':
    print("--- Starting Pipeline V2 Initialization ---")
    for archive in TARGET_ARCHIVES:
        _extract_archive(archive, WORKING_DIR)

    all_parquets: List[str] = glob.glob(f"{WORKING_DIR}/**/*.parquet", recursive=True)
    if not all_parquets:
        raise FileNotFoundError(f"No .parquet files found in {WORKING_DIR}. Check target archives.")

    market_directories: List[str] = list(set([os.path.dirname(p) for p in all_parquets]))
    print(f"\nDiscovered {len(market_directories)} uncompressed markets. Beginning parallel V2 generation...")

    with concurrent.futures.ProcessPoolExecutor() as executor:
        results: List[str] = list(executor.map(process_single_market, market_directories))

    for r in results:
        print(r)

    print(f"\nExtraction Complete. Ready for Colab Phase 2.")

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from typing import List
from sklearn.model_selection import train_test_split
import glob

class KalshiDenseDataset(Dataset):
    def __init__(self, file_paths: List[str], seq_len: int = 60, forward_look: int = 30,
                 profit_target: float = 0.04, stop_loss: float = 0.02, stride: int = 2,
                 inference_mode: bool = False):
        """
        Triple Barrier Method Dataset for Short (NO) Taker Strategy.
        Target Classes:
        2: Upper Barrier Hit (Profit - Market Crashed)
        1: Time Barrier Hit (Flat / No Action)
        0: Lower Barrier Hit (Stopped Out - Market Rallied)
        """
        self.seq_len = seq_len
        self.forward_look = forward_look
        self.profit_target = profit_target
        self.stop_loss = stop_loss
        self.stride = stride
        self.inference_mode = inference_mode

        self.bid_col = 'bid_px_1'
        self.ask_col = 'ask_px_1' # NEW
        self.trade_col = 'window_min_trade_price'
        self.trade_max_col = 'window_max_trade_price'

        print(f"Loading {len(file_paths)} parquet files...")
        dfs = []
        for f in file_paths:
            df = pd.read_parquet(f)
            df['ticker'] = os.path.basename(f) # INJECT TICKER TO PREVENT BLEEDING
            dfs.append(df)

        self.data = pd.concat(dfs, ignore_index=True)

        # --- NEW: ORDER FLOW IMBALANCE (OFI) ---
        # 1. Bid Side OFI
        bid_px = self.data['bid_px_1']
        bid_vol = self.data['bid_vol_1']
        bid_px_prev = bid_px.shift(1)
        bid_vol_prev = bid_vol.shift(1)

        bid_ofi = np.zeros(len(self.data))
        bid_ofi = np.where(bid_px > bid_px_prev, bid_vol, bid_ofi)
        bid_ofi = np.where(bid_px == bid_px_prev, bid_vol - bid_vol_prev, bid_ofi)
        bid_ofi = np.where(bid_px < bid_px_prev, -bid_vol_prev, bid_ofi)

        # 2. Ask Side OFI
        ask_px = self.data['ask_px_1']
        ask_vol = self.data['ask_vol_1']
        ask_px_prev = ask_px.shift(1)
        ask_vol_prev = ask_vol.shift(1)

        ask_ofi = np.zeros(len(self.data))
        ask_ofi = np.where(ask_px < ask_px_prev, ask_vol, ask_ofi)
        ask_ofi = np.where(ask_px == ask_px_prev, ask_vol - ask_vol_prev, ask_ofi)
        ask_ofi = np.where(ask_px > ask_px_prev, -ask_vol_prev, ask_ofi)

        # 3. Net OFI (Symmetrically Log-Scaled to handle extreme pressure spikes)
        raw_ofi = bid_ofi - ask_ofi
        self.data['OFI'] = np.sign(raw_ofi) * np.log1p(np.abs(raw_ofi))
        self.data['OFI'] = self.data['OFI'].fillna(0.0) # Handle shift(1) NaN

        # --- NEW: Log-Normalize Volumes ---
        # Dynamically find all volume columns (bid_vol_X, ask_vol_X, taker_vol_yes, etc.)
        vol_cols: List[str] = [col for col in self.data.columns if 'vol' in col.lower()]

        # Apply natural log transformation: log(1 + x)
        self.data[vol_cols] = np.log1p(self.data[vol_cols].astype(float))

        self.ticker_series: np.ndarray = self.data['ticker'].values

        self.ts_array: np.ndarray = self.data['ts'].values


        drop_cols = ['ticker', 'ts', 'local_recv_ts', 'window_min_trade_price', 'window_max_trade_price']
        self.features_array = self.data.drop(columns=drop_cols, errors='ignore').values
        self.features_array = np.nan_to_num(self.features_array, nan=0.0, posinf=0.0, neginf=0.0)

        self.bid_array = self.data[self.bid_col].values
        self.ask_array = self.data[self.ask_col].values # NEW

        if self.trade_col in self.data.columns:
            self.trade_min_array = self.data[self.trade_col].fillna(np.inf).values
            self.trade_max_array = self.data[self.trade_max_col].fillna(-np.inf).values # NEW
        else:
            raise ValueError("Trade columns missing.")

        # Identify valid start indices
        valid_indices = []
        max_idx = len(self.data) - self.forward_look - 1

        for i in range(self.seq_len, max_idx, self.stride):
            # Ensure the sequence and forward-look window do not cross ticker boundaries
            if self.ticker_series[i - self.seq_len] == self.ticker_series[i + self.forward_look]:
                valid_indices.append(i)

        self.indices = valid_indices
        print(f"Dataset built with {len(self.indices)} valid samples.")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        current_idx = self.indices[idx]

        # 1. Extract historical sequence
        history_window = self.features_array[current_idx - self.seq_len : current_idx]
        features = torch.tensor(history_window, dtype=torch.float32)

        if self.inference_mode:
            ticker = str(self.ticker_series[current_idx])
            ts = float(self.ts_array[current_idx]) # Adjust variable names if yours differ slightly
            return ticker, ts, features

        # 2. Define Entry Prices
        p_bid = self.bid_array[current_idx]
        p_ask = self.ask_array[current_idx]

        exhausted_bid = False
        exhausted_ask = False
        ask_repriced_down = False
        bid_repriced_up = False

        # 3. Look forward 30 seconds
        for step in range(1, self.forward_look + 1):
            future_idx = current_idx + step

            future_min_trade = self.trade_min_array[future_idx]
            future_max_trade = self.trade_max_array[future_idx]
            future_bid = self.bid_array[future_idx]
            future_ask = self.ask_array[future_idx]

            # Did the taker completely eat our queue?
            if future_min_trade < p_bid:
                exhausted_bid = True
            if future_max_trade > p_ask:
                exhausted_ask = True

            # Did the market officially shift against our hypothetical position?
            if future_ask < p_ask:
                ask_repriced_down = True
            if future_bid > p_bid:
                bid_repriced_up = True

            if exhausted_bid and exhausted_ask:
                break # Whipsaw achieved, stop checking

        # 4. Your Partition Logic
        if exhausted_bid and exhausted_ask:
            label = 2  # Profit: True Whipsaw

        elif (exhausted_bid and not exhausted_ask and ask_repriced_down) or \
             (exhausted_ask and not exhausted_bid and bid_repriced_up):
            label = 0  # Loss: Legged out AND the market repriced against us

        else:
            label = 1  # Flat: Either no fills, or legged but we can still scratch the trade

        return features, torch.tensor(label, dtype=torch.long)

In [ ]:
import os
import glob
from sklearn.model_selection import train_test_split

print("Scanning local disk for Dense Tensors...")
# Point this to wherever your generated files are saved
local_path = '/content/kalshi_dense_state_v2'
all_dense_files = glob.glob(f"{local_path}/*.parquet")

if not all_dense_files:
    raise FileNotFoundError(f"No parquet files found in {local_path}.")

# Group files by Base Game to prevent data leakage
market_groups = {}
for file_path in all_dense_files:
    filename = os.path.basename(file_path)
    # Splits 'dense_KXMLBGAME-26APR191510LADCOL-LAD.parquet'
    parts = filename.split('-')
    if len(parts) >= 2:
        base_game = parts[1] # Extracts '26APR191510LADCOL'
        if base_game not in market_groups:
            market_groups[base_game] = []
        market_groups[base_game].append(file_path)

unique_games = list(market_groups.keys())
print(f"Found {len(unique_games)} unique MLB games.")

# Split 80/20 strictly by GAME, not by individual file
train_games, val_games = train_test_split(unique_games, test_size=0.2, random_state=42)

# Flatten back into lists of file paths for the Dataset class
train_files = [f for game in train_games for f in market_groups[game]]
val_files = [f for game in val_games for f in market_groups[game]]

print(f"Train Set: {len(train_games)} games -> {len(train_files)} dual-market files")
print(f"Val Set: {len(val_games)} games -> {len(val_files)} dual-market files")



# --- Instantiate DataLoaders ---

print("\nBuilding Training Dataset (Triple Barrier NO-Strategy)...")
train_dataset = KalshiDenseDataset(
    file_paths=train_files,
    seq_len=60,
    forward_look=30,
    profit_target=0.04,
    stop_loss=0.02,
    stride=2,
    inference_mode=False
)

print("\nBuilding Validation Dataset (Triple Barrier NO-Strategy)...")
val_dataset = KalshiDenseDataset(
    file_paths=val_files,
    seq_len=60,
    forward_look=30,
    profit_target=0.04,
    stop_loss=0.02,
    stride=2,
    inference_mode=False
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nDataLoaders Ready.")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe: Tensor = torch.zeros(max_len, d_model)
        position: Tensor = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term: Tensor = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return x

class KalshiTransformer(nn.Module):
    def __init__(self, feature_dim: int = 23, d_model: int = 64, nhead: int = 4, num_layers: int = 2, num_classes: int = 3):
        super().__init__()
        self.d_model: int = d_model


        # 1. Input Projection
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # 2. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dropout=0.1)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 3. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, num_classes)
        )

    def forward(self, x: Tensor) -> Tensor:

        # Project and encode
        x = self.input_proj(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # Pass through attention layers
        x = self.transformer_encoder(x)

        # Extract the final time-step representation
        final_step: Tensor = x[:, -1, :]

        logits: Tensor = self.classifier(final_step)
        return logits

class FocalLoss(nn.Module):
    def __init__(self, alpha: Tensor, gamma: float = 2.0):
        super().__init__()
        self.alpha: Tensor = alpha
        self.gamma: float = gamma

    def forward(self, inputs: Tensor, targets: Tensor) -> Tensor:
        ce_loss: Tensor = F.cross_entropy(inputs, targets, reduction='none')
        pt: Tensor = torch.exp(-ce_loss)
        alpha_t: Tensor = self.alpha.gather(0, targets)
        focal_loss: Tensor = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from tqdm import tqdm

# --- 1. SETUP MODEL, OPTIMIZER, AND LOSS ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# Instantiate your Transformer
model = KalshiTransformer(feature_dim=24, d_model=64, nhead=4, num_layers=2, num_classes=3).to(device)

# Standard AdamW optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Initialize Focal Loss with the EXACT weights from your Colab audit
alpha_weights = torch.tensor([0.15299715101718903, 0.6851131916046143, 0.16188965737819672]).to(device)
criterion = FocalLoss(alpha=alpha_weights, gamma=2.0).to(device)

best_val_loss: float = float('inf')
best_model_path: str = 'best_alpha_model.pth'

# --- 2. TRAINING LOOP ---
epochs = 5

for epoch in range(epochs):
    print(f"\n{'='*40}")
    print(f"EPOCH {epoch+1} / {epochs}")
    print(f"{'='*40}")

    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0

    # Optional: wrapping dataloader in tqdm for progress bar
    train_bar = tqdm(train_loader, desc="Training", leave=False)
    for batch_features, batch_labels in train_bar:
        # Move to GPU
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # Forward Pass
        optimizer.zero_grad()
        logits = model(batch_features)

        # Compute Loss
        loss = criterion(logits, batch_labels)

        # Backward Pass & Optimize
        loss.backward()

        # Gradient clipping (Critical for Transformers to prevent exploding gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        train_loss += loss.item()
        train_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0

    all_preds = []
    all_targets = []

    val_bar = tqdm(val_loader, desc="Validating", leave=False)
    with torch.no_grad():
        for batch_features, batch_labels in val_bar:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_features)
            loss = criterion(logits, batch_labels)
            val_loss += loss.item()

            # Get predictions (argmax of logits)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(batch_labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    # --- METRICS & PRINTOUTS ---
    print(f"\nTrain Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # Generate Confusion Matrix
    cm = confusion_matrix(all_targets, all_preds, labels=[0, 1, 2])
    print("\nConfusion Matrix (Rows: True, Cols: Predicted [0:Loss, 1:Flat, 2:Profit]):")
    print(cm)

    # Generate detailed precision/recall report
    # zero_division=0 prevents warnings if the model predicts 0 instances of a class
    report = classification_report(all_targets, all_preds, target_names=['0 (Loss)', '1 (Flat)', '2 (Profit)'], zero_division=0)
    print("\nClassification Report:")
    print(report)

    if avg_val_loss < best_val_loss:
        print(f"\n[+] Validation loss improved from {best_val_loss:.4f} to {avg_val_loss:.4f}. Saving model to {best_model_path}!")
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), best_model_path)

print("\nTraining Complete.")

The simulator code begins here.  An attempt at a near perfect market simulator to efficiently and accurately test and iterate model architecture and execution logic.  Execution logic is also kept below.

In [ ]:
# --- CONFIGURATION ---
CONFIDENCE_THRESHOLD = 0.4
OUTPUT_PREDS_PATH = 'predictions.parquet'

def run_inference_engine(model: torch.nn.Module, dataloader: DataLoader):
    model.load_state_dict(torch.load('best_alpha_model.pth', weights_only=True))
    model.eval()
    all_predictions = []

    print(f"Executing forward passes at Threshold: {CONFIDENCE_THRESHOLD}...")

    with torch.no_grad():
        # THE FIX: Unpack EXACTLY 3 variables (tickers, ts_batch, x_batch)
        for tickers, ts_batch, x_batch in dataloader:
            x_batch = x_batch.to(device)

            logits = model(x_batch)
            probs = torch.nn.functional.softmax(logits, dim=1)

            confidences, predicted_classes = torch.max(probs, dim=1)

            confidences = confidences.cpu().numpy()
            predicted_classes = predicted_classes.cpu().numpy()

            # --- UPDATED MAKER-MAKER MAPPING ---
            # 0: Toxic Loss (Avoid), 1: Noise (Avoid), 2: Whipsaw (Trade!)
            side_map = {0: 'ignore', 1: 'ignore', 2: 'maker'}

            for i in range(len(tickers)):
                conf = confidences[i]
                pred_side = side_map[predicted_classes[i]]

                # STRICT FILTER: Only log signals when the model predicts a Whipsaw
                if pred_side == 'maker' and conf >= CONFIDENCE_THRESHOLD:
                    ts_val = ts_batch[i].item() if torch.is_tensor(ts_batch[i]) else ts_batch[i]
                    all_predictions.append({
                        'ticker': tickers[i],
                        'ts': ts_val,
                        # Pass ts as local_recv_ts so the Simulator's FSM doesn't crash looking for the column
                        'local_recv_ts': ts_val,
                        'predicted_side': pred_side,
                        'confidence': conf
                    })

    df_preds = pd.DataFrame(all_predictions)

    if not df_preds.empty:
        df_preds = df_preds.sort_values(by='local_recv_ts').reset_index(drop=True)
        df_preds.to_parquet(OUTPUT_PREDS_PATH, index=False)
        print(f"\n--- Engine Complete ---")
        print(f"Saved {len(df_preds)} actionable Maker-Maker signals to {OUTPUT_PREDS_PATH}.")
        print(df_preds.head())
    else:
        print("\n--- Engine Complete ---")
        print(f"No signals met the confidence threshold of {CONFIDENCE_THRESHOLD}.")

if __name__ == '__main__':
    print("Building Inference Dataset...")

    val_dataset_inference = KalshiDenseDataset(
        file_paths=val_files,
        seq_len=60,
        forward_look=30,
        stride=5,
        inference_mode=True
    )

    val_loader_inference = DataLoader(val_dataset_inference, batch_size=256, shuffle=False)

    run_inference_engine(model, val_loader_inference)

In [ ]:
import pandas as pd
from typing import Dict, Optional, Any, Tuple

class KalshiMatchingEngine:
    def __init__(self) -> None:
        self.resting_bids: Dict[float, float] = {}
        self.resting_asks: Dict[float, float] = {}
        self.last_snapshot_seq: int = -1
        #MAKER UPGRADE
        self.active_orders: Dict[str, Optional[Dict[str, Any]]] = {
            'yes': None, # Our resting Bid
            'no': None   # Our resting Ask
        }

    def get_best_bid(self) -> Optional[float]:
        valid_bids = [px for px, vol in self.resting_bids.items() if vol > 1e-5]
        best_raw = max(valid_bids) if valid_bids else None

        my_order = self.active_orders.get('yes')
        if my_order and my_order['receipt']['status'] == 'pending':
            my_px = my_order['order_price']
            return max(best_raw, my_px) if best_raw else my_px
        return best_raw

    def get_best_ask(self) -> Optional[float]:
        valid_asks = [px for px, vol in self.resting_asks.items() if vol > 1e-5]
        best_raw = min(valid_asks) if valid_asks else None

        my_order = self.active_orders.get('no')
        if my_order and my_order['receipt']['status'] == 'pending':
            my_px = my_order['order_price']
            return min(best_raw, my_px) if best_raw else my_px
        return best_raw

    def process_tick(self, row: dict) -> None:
        """The master method called on every tick to maintain the universe."""
        event_type: str = str(row.get('event_type'))

        if event_type not in ['orderbook_snapshot', 'orderbook_delta', 'trade']:
            return

        # --- DYNAMIC SCHEMA ROUTING ---
        if event_type == 'trade':
            raw_price = row.get('yes_price_dollars')
            if raw_price is None or pd.isna(raw_price):
                return
            price: float = float(raw_price)
            tick_side: str = str(row.get('taker_side'))

        else: # orderbook_snapshot or orderbook_delta
            raw_price = row.get('price_dollars')
            tick_side: str = str(row.get('side'))
            if raw_price is None or pd.isna(raw_price) or tick_side not in ['yes', 'no']:
                return
            # THE FIX: Force strict 2-decimal rounding on the dictionary keys
            price: float = round(float(raw_price), 2) if tick_side == 'yes' else round(1.0 - float(raw_price), 2)

        # 1. Maintain the LOB
        if row['event_type'] in ['orderbook_snapshot', 'orderbook_delta']:
            book = self.resting_bids if tick_side == 'yes' else self.resting_asks

            if row['event_type'] == 'orderbook_snapshot':
                if row['seq'] != self.last_snapshot_seq:
                    self.resting_bids.clear()
                    self.resting_asks.clear()
                    self.last_snapshot_seq = row['seq']

            delta: float = float(row['delta_fp']) if pd.notna(row['delta_fp']) else 0.0
            new_vol: float = book.get(price, 0.0) + delta

            if new_vol > 1e-5:
                book[price] = new_vol
            else:
                book.pop(price, None)

        # --- THE MAKER-MAKER FIX: Loop through both active orders ---
        for o_side, order in self.active_orders.items():
            if order is None or order['receipt']['status'] != 'pending':
                continue

            # 2. Queue Physics: Pulls
            if row['event_type'] in ['orderbook_snapshot', 'orderbook_delta']:
                if tick_side == o_side and price == order['order_price']:
                    if delta < 0:
                        pulled: float = abs(delta)
                        order['v_ahead'] = max(0.0, order['v_ahead'] - (pulled * 0.5))
                        order['receipt']['pulls_while_waiting'] += pulled

            # 3. Queue Physics: Trades
            elif event_type == 'trade':
                if float(row['local_recv_ts']) <= order['receipt']['arrival_ts']:
                    continue
                traded: float = float(row.get('count_fp', 0.0))

                if tick_side != o_side:
                    is_hit = False
                    if o_side == 'yes' and price <= order['order_price']:
                        is_hit = True
                    elif o_side == 'no' and price >= order['order_price']:
                        is_hit = True

                    if is_hit:
                        order['receipt']['trades_ahead_of_us'] += traded

                        # A. Taker volume must eat the queue ahead of us first
                        if order['v_ahead'] > 0:
                            consumed = min(order['v_ahead'], traded)
                            order['v_ahead'] -= consumed
                            traded -= consumed

                        # B. If we are at the front, the Taker volume finally eats OUR order
                        if order['v_ahead'] <= 1e-5 and traded > 0:
                            order['my_volume'] -= traded
                            if order['my_volume'] <= 1e-5:
                                order['receipt']['status'] = 'filled'
                                order['receipt']['fill_ts'] = float(row.get('local_recv_ts'))
                                order['receipt']['fill_price'] = order['order_price']

    def submit_limit_order(self, side: str, price: float, arrival_ts: float, signal_ts: float, volume: float = 1.0) -> None:
        """Places an independent order into the YES or NO queue."""
        if side not in ['yes', 'no']:
            return

        order_price = round(price, 2)
        book = self.resting_bids if side == 'yes' else self.resting_asks
        v_ahead = book.get(order_price, 0.0)

        # Store the order state entirely within its specific side's dictionary
        self.active_orders[side] = {
            'order_price': order_price,
            'my_volume': volume,
            'v_ahead': v_ahead,
            'receipt': {
                'signal_ts': signal_ts,
                'arrival_ts': arrival_ts,
                'target_price': order_price,
                'initial_v_ahead': v_ahead,
                'pulls_while_waiting': 0.0,
                'trades_ahead_of_us': 0.0,
                'fill_ts': None,
                'status': 'pending'
            }
        }

    def cancel_active_order(self, side: str, cancel_ts: float) -> Optional[Dict[str, Any]]:
        """Cancels an order on a specific side and returns its receipt."""
        order = self.active_orders.get(side)

        if order is not None and order['receipt']['status'] == 'pending':
            order['receipt']['status'] = 'canceled'
            order['receipt']['exit_ts'] = cancel_ts

            # Copy the receipt to return it, then clear the slot
            final_receipt = order['receipt'].copy()
            self.active_orders[side] = None

            return final_receipt

        return None

In [ ]:
class KalshiExecutor:
    def __init__(self, predictions_df: pd.DataFrame):
        # Load and sort predictions chronologically
        self.predictions = predictions_df.sort_values(by='local_recv_ts').to_dict('records')
        self.pred_idx = 0

        # FSM State
        self.state = 'WAITING_FOR_SIGNAL'
        self.active_signal = None
        self.filled_leg = None
        self.entry_receipt = None

        # --- MAKER-MAKER HYPERPARAMETERS ---
        self.LATENCY_PENALTY = 0.020 # 20 milliseconds
        self.WHIPSAW_TIMEOUT = 30.0  # Time to wait for both legs to fill

        # Active Inventory Management (The Chase)
        self.PEG_CYCLE_SEC = 3.0       # Update order every 3 seconds
        self.TTL_PUKE_SEC = 60.0       # 60s chase max (20 attempts) to prevent orphaned backtest trades
        self.MIN_SPREAD_BAILOUT = 0.01 # Bailout if spread compresses to 1 tick

        self.whipsaw_start_ts = None
        self.peg_start_ts = None
        self.last_peg_ts = None
        # -----------------------------------

        # Final Audit Log
        self.completed_trades = []

    def update(self, row: pd.Series, engine: KalshiMatchingEngine) -> None:
        """The FSM tick loop. Evaluates exactly one state per microsecond."""
        current_ts = float(row['local_recv_ts'])

        # ---------------------------------------------------------
        # STATE 1: WAITING FOR SIGNAL
        # ---------------------------------------------------------
        if self.state == 'WAITING_FOR_SIGNAL':
            if self.pred_idx >= len(self.predictions):
                return

            next_signal = self.predictions[self.pred_idx]

            if current_ts >= next_signal['local_recv_ts'] + self.LATENCY_PENALTY:
                self.active_signal = next_signal
                self.pred_idx += 1

                best_bid = engine.get_best_bid()
                best_ask = engine.get_best_ask()

                # Require a 2c spread so the Arb is actually profitable after fees
                if best_bid is None or best_ask is None or (best_ask - best_bid) < 0.02:
                    self._log_scratch("Skipped - Vacuum or Spread < 2c")
                    return

                # Deploy the Maker-Maker Net!
                engine.submit_limit_order('yes', best_bid, current_ts, self.active_signal['local_recv_ts'])
                engine.submit_limit_order('no', best_ask, current_ts, self.active_signal['local_recv_ts'])

                self.state = 'MAKER_PENDING'
                self.whipsaw_start_ts = current_ts

        # ---------------------------------------------------------
        # STATE 2: MAKER PENDING (Waiting for the Whipsaw)
        # ---------------------------------------------------------
        elif self.state == 'MAKER_PENDING':
            yes_order = engine.active_orders.get('yes')
            no_order = engine.active_orders.get('no')

            yes_filled = yes_order and yes_order['receipt']['status'] == 'filled'
            no_filled = no_order and no_order['receipt']['status'] == 'filled'

            # 1. THE HOLY GRAIL: Both sides filled!
            if yes_filled and no_filled:
                entry_px = yes_order['receipt']['fill_price']
                exit_px = no_order['receipt']['fill_price']
                pnl = round(exit_px - entry_px, 2)

                self.entry_receipt = yes_order['receipt']
                self._log_closed_trade("Win - Full Whipsaw Arb", exit_px, current_ts, pnl)
                return

            # 2. Timeout: 30 seconds have passed. Assess the damage.
            if current_ts - self.whipsaw_start_ts > self.WHIPSAW_TIMEOUT:

                # Only cancel if BOTH legs are dead.
                if not yes_filled and not no_filled:
                    engine.cancel_active_order('yes', current_ts)
                    engine.cancel_active_order('no', current_ts)
                    self._log_scratch("Scratch - Dead Market, No Fills")
                    return

                # We are legged! Move to Active Pegging.
                # DO NOT cancel the unfilled leg here. We need it to hold our queue position!
                self.filled_leg = 'yes' if yes_filled else 'no'
                self.entry_receipt = yes_order['receipt'] if yes_filled else no_order['receipt']

                self.state = 'ACTIVE_PEGGING'
                self.peg_start_ts = current_ts
                self.last_peg_ts = current_ts - self.PEG_CYCLE_SEC # Force immediate peg drop on next tick

        # ---------------------------------------------------------
        # STATE 3: ACTIVE PEGGING (The Legged Chase)
        # ---------------------------------------------------------
        elif self.state == 'ACTIVE_PEGGING':
            exit_side = 'no' if self.filled_leg == 'yes' else 'yes'
            exit_order = engine.active_orders.get(exit_side)

            entry_px = self.entry_receipt['fill_price']
            time_in_peg = current_ts - self.peg_start_ts

            # A. Did our peg get hit naturally?
            if exit_order and exit_order['receipt']['status'] == 'filled':
                exit_px = exit_order['receipt']['fill_price']
                pnl = exit_px - entry_px if self.filled_leg == 'yes' else entry_px - exit_px
                self._log_closed_trade("Scratch/Loss - Peg Hit", exit_px, current_ts, round(pnl, 2))
                return

            # B. Time-To-Live (TTL) Hard Puke (60s Limit Hit)
            if time_in_peg > self.TTL_PUKE_SEC:
                engine.cancel_active_order(exit_side, current_ts)
                exit_px, pnl = self._execute_market_exit(engine, entry_px, self.filled_leg)
                self._log_closed_trade("Loss - 60s Chase Timeout", exit_px, current_ts, pnl)
                return

            # C. The 3-Second Pegging Cycle (WITH QUEUE RETENTION)
            if current_ts - self.last_peg_ts >= self.PEG_CYCLE_SEC:
                self.last_peg_ts = current_ts

                best_bid = engine.get_best_bid()
                best_ask = engine.get_best_ask()

                if best_bid is None or best_ask is None:
                    return # Market vacuum, wait for next tick

                current_spread = round(best_ask - best_bid, 2)

                # Spread Bailout: If spread is strictly 1c, the chase is over.
                if current_spread <= self.MIN_SPREAD_BAILOUT:
                    engine.cancel_active_order(exit_side, current_ts)
                    exit_px, pnl = self._execute_market_exit(engine, entry_px, self.filled_leg)
                    self._log_closed_trade("Loss/Scratch - 1c Spread Bailout", exit_px, current_ts, pnl)
                    return

                # --- TOP OF BOOK QUEUE RETENTION ---
                current_order_px = exit_order['order_price'] if exit_order and exit_order['receipt']['status'] == 'pending' else None

                if self.filled_leg == 'yes':
                    # We are Long YES, need to sell. Are we already the lowest Ask?
                    if current_order_px is not None and current_order_px <= best_ask:
                        return # Leave it alone! Retain queue priority.

                    target_px = round(best_ask - 0.01, 2)
                    if target_px <= best_bid: target_px = round(best_bid + 0.01, 2)
                else:
                    # We are Short YES, need to buy. Are we already the highest Bid?
                    if current_order_px is not None and current_order_px >= best_bid:
                        return # Leave it alone! Retain queue priority.

                    target_px = round(best_bid + 0.01, 2)
                    if target_px >= best_ask: target_px = round(best_ask - 0.01, 2)

                # We are NOT at the top of the book. Cancel and replace to chase.
                engine.cancel_active_order(exit_side, current_ts)
                engine.submit_limit_order(exit_side, target_px, current_ts, current_ts)

    # --- Audit Logging Helpers ---
    def _log_scratch(self, reason: str):
        """Logs signals that never resulted in a position."""
        self.completed_trades.append({
            'ticker': self.active_signal['ticker'],
            'signal_side': self.active_signal.get('predicted_side', 'maker'),
            'confidence': self.active_signal.get('confidence'),
            'outcome': reason,
            'entry_price': None,
            'exit_price': None,
            'gross_pnl': 0.0
        })
        self._reset_fsm()

    def _log_closed_trade(self, exit_reason: str, exit_px: float, exit_ts: float, pnl: float):
        """Logs completed round-trip trades."""
        self.completed_trades.append({
            'ticker': self.active_signal['ticker'],
            'signal_side': self.active_signal.get('predicted_side', 'maker'),
            'confidence': self.active_signal.get('confidence'),
            'outcome': exit_reason,
            'entry_price': self.entry_receipt['fill_price'],
            'exit_price': exit_px,
            'time_in_trade': exit_ts - self.entry_receipt['fill_ts'],
            'exit_ts': exit_ts,
            'gross_pnl': pnl
        })
        self._reset_fsm()

    def _reset_fsm(self):
        self.state = 'WAITING_FOR_SIGNAL'
        self.active_signal = None
        self.entry_receipt = None
        self.filled_leg = None
        self.whipsaw_start_ts = None
        self.peg_start_ts = None
        self.last_peg_ts = None

    def _execute_market_exit(self, engine: KalshiMatchingEngine, entry_px: float, filled_leg: str):
        """Crosses the spread and calculates true PnL from the Entry Price."""
        if filled_leg == 'yes':
            # We are Long YES. We must sell to the Best Bid.
            exit_px = engine.get_best_bid()
            if exit_px is None: exit_px = 0.01
            pnl = exit_px - entry_px
        else:
            # We are Short YES. We must buy from the Best Ask.
            exit_px = engine.get_best_ask()
            if exit_px is None: exit_px = 0.99
            pnl = entry_px - exit_px

        return round(exit_px, 2), round(pnl, 2)

In [ ]:
import pandas as pd
import gcsfs
from tqdm import tqdm
import time

# --- CONFIGURATION ---
PREDICTIONS_PATH = 'predictions.parquet'
RAW_BUCKET_PATH = 'tryanheider-kalshi-bucket/kalshi_clean_raw_v2'
OUTPUT_AUDIT_PATH = 'trade_receipts.parquet'

print("=== INITIATING ZERO-ASSUMPTION SIMULATOR ===")

# 1. Load the Signals
df_preds = pd.read_parquet(PREDICTIONS_PATH)
tickers = df_preds['ticker'].unique()
print(f"Loaded {len(df_preds)} signals across {len(tickers)} unique markets.")

# 2. Connect to the Raw Vault
fs = gcsfs.GCSFileSystem()
all_receipts = []

start_time = time.time()

# 3. The Master Loop
for ticker in tqdm(tickers, desc="Simulating Markets"):
    # Isolate signals for this specific market
    ticker_preds = df_preds[df_preds['ticker'] == ticker].copy()

    # --- THE STRING FIX ---
    # Strip the "dense_" prefix and ".parquet" suffix to get the true Kalshi ticker
    clean_ticker = ticker.replace("dense_", "").replace(".parquet", "")

    # Load the pure, un-shredded tick data
    raw_file_path = f"gs://{RAW_BUCKET_PATH}/clean_raw_{clean_ticker}.parquet"

    try:
        df_raw = pd.read_parquet(raw_file_path)
    except FileNotFoundError:
        print(f"\n[!] Raw data missing for {clean_ticker}. Skipping.")
        continue

    # Instantiate the clean-slate Engine and FSM Executor
    engine = KalshiMatchingEngine()
    executor = KalshiExecutor(ticker_preds)

    # Fast iteration: Convert DataFrame to a list of Python dictionaries
    raw_records = df_raw.to_dict('records')

    # The microsecond tick loop
    for row in raw_records:
        engine.process_tick(row)
        executor.update(row, engine)

    # Harvest the FSM's audit log for this market
    all_receipts.extend(executor.completed_trades)

# 4. Generate the Final Audit Log
end_time = time.time()
df_audit = pd.DataFrame(all_receipts)

print("\n=== SIMULATION COMPLETE ===")
print(f"Execution Time: {round(end_time - start_time, 2)} seconds")

if not df_audit.empty:
    df_audit.to_parquet(OUTPUT_AUDIT_PATH, index=False)

    print(f"\nTotal Signals Processed: {len(df_audit)}")
    print("\n--- OUTCOME DISTRIBUTION ---")
    print(df_audit['outcome'].value_counts())

    print("\n--- PNL SUMMARY ---")
    gross_pnl = df_audit['gross_pnl'].sum()
    print(f"Total Gross PnL: ${gross_pnl:.2f}")

    # Display the first few actionable trades
    filled_trades = df_audit[~df_audit['outcome'].str.contains('Skipped|Scratch')]
    if not filled_trades.empty:
        print("\n--- SAMPLE COMPLETED TRADES ---")
        print(filled_trades.head())
else:
    print("\n[!] No trades were logged. Check the FSM logic or threshold constraints.")